# Fig. 3(b) Reproduction With `bloqade-decoders` and Paper-Style Data Flow

This notebook mirrors the **paper-side decoding pipeline** more closely than `msd_reprod_bloqade_decoders.ipynb` while still using the Gemini Logical MVP compilation flow in `bloqade-lanes`.

What is paper-faithful here:
- MLD is trained on a **special Clifford surrogate circuit** sampled with the detector sampler.
- MLD postselection patterns are ranked using **separate actual MSD data**, not the surrogate training data.
- Injection fidelity is computed with a **separate injection decoder / dataset split**.
- MLE uses the `bloqade-decoders` Gurobi decoder with **logical-gap** scoring.

What is still Bloqade-specific:
- The physical schedule and noise insertion come from the Gemini Logical compiler and the `bloqade-lanes` neutral-atom noise hooks, not the hand-written `distillation_sim` pulse/move schedule.
- The notebook uses the local `demo/msd_utils/noise.py` approximation of the `distillation_sim` noise parameters.
        


In [1]:
import itertools
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
for candidate in PROJECT_ROOT_CANDIDATES:
    candidate = candidate.resolve()
    if (candidate / 'demo' / 'msd_utils').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('Could not locate repo root containing demo/msd_utils.')

LOCAL_DECODER_SRC_CANDIDATES = [
    Path.cwd() / '..' / 'bloqade-decoders' / 'src',
    Path.cwd() / 'bloqade-decoders' / 'src',
    Path.cwd().parent / 'bloqade-decoders' / 'src',
    Path.cwd().parent.parent / 'bloqade-decoders' / 'src',
]
for candidate in LOCAL_DECODER_SRC_CANDIDATES:
    candidate = candidate.resolve()
    if candidate.exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('Could not locate sibling bloqade-decoders/src checkout.')

import bloqade.decoders as bloqade_decoders
from bloqade.decoders import GurobiDecoder, TableDecoder
from bloqade.lanes import GeminiLogicalSimulator
from bloqade.lanes.steane_defaults import steane7_m2dets, steane7_m2obs

from demo.msd_utils.core import (
    DEFAULT_BASIS_LABELS,
    DEFAULT_TARGET_BLOCH,
    logical_expectation,
    run_task,
    split_factory_bits,
    infer_factory_target,
    infer_distilled_sign_vector,
)
from demo.msd_utils.circuits import (
    build_paper_decoder_kernel_bundle,
    build_task_map,
    make_noisy_steane7_initializer,
)
from demo.msd_utils.decoders import (
    build_mld_decoders,
    build_mle_decoders,
    evaluate_curve,
    evaluate_mld_curve,
    injected_baseline,
)
from demo.msd_utils.noise import (
    DistillationSimNoiseConfig,
    build_distillation_sim_noise_model,
)

print('Using bloqade.decoders from:', Path(bloqade_decoders.__file__).resolve())
        


Using bloqade.decoders from: /Users/jasonhan/Desktop/qmain/kirin-workspace/bloqade-decoders/src/bloqade/decoders/__init__.py


In [2]:
FAST_CONFIG = {
    'mld_train_shots': 20_000,
    'mld_score_shots': 8_000,
    'eval_shots': 8_000,
    'injection_train_shots': 20_000,
    'injection_eval_shots': 8_000,
    'target_inference_shots': 10_000,
    'posterior_samples': 20_000,
    'mle_threshold_points': 32,
}

PAPER_SCALE_CONFIG = {
    'mld_train_shots': 1_000_000,
    'mld_score_shots': 250_000,
    'eval_shots': 250_000,
    'injection_train_shots': 1_000_000,
    'injection_eval_shots': 250_000,
    'target_inference_shots': 50_000,
    'posterior_samples': 200_000,
    'mle_threshold_points': 64,
}

CONFIG = FAST_CONFIG.copy()

IDEAL_THETA = 0.3041 * math.pi
IDEAL_PHI = 0.25 * math.pi
IDEAL_LAM = 0.0
THETA_OFFSET = 0.30
PHI_OFFSET = 0.0
LAM_OFFSET = 0.0

THETA = IDEAL_THETA + THETA_OFFSET
PHI = IDEAL_PHI + PHI_OFFSET
LAM = IDEAL_LAM + LAM_OFFSET

BASIS_LABELS = DEFAULT_BASIS_LABELS
OUTPUT_LOGICAL = 3
CHECKS_PER_LOGICAL = 3
MLE_SCORE_MODE = 'logical_gap'
MAGIC_TARGET_BLOCH = DEFAULT_TARGET_BLOCH


def u3_prep_bloch(theta: float, phi: float) -> np.ndarray:
    return np.array([
        math.sin(theta) * math.cos(phi),
        math.sin(theta) * math.sin(phi),
        math.cos(theta),
    ], dtype=np.float64)


PREP_BLOCH = u3_prep_bloch(THETA, PHI)
M2DETS_5 = steane7_m2dets(5)
M2OBS_5 = steane7_m2obs(5)
M2DETS_1 = steane7_m2dets(1)
M2OBS_1 = steane7_m2obs(1)
        


In [ ]:
def infer_single_logical_sign_vector(task_map, *, shots, target_bloch):
    raw_bits = {}
    for basis in BASIS_LABELS:
        data = run_task(task_map[basis], shots, with_noise=False, run_detectors=False)
        raw_bits[basis] = data.observables[:, 0].astype(np.uint8)

    raw_bloch = np.array(
        [logical_expectation(raw_bits[basis]) for basis in BASIS_LABELS],
        dtype=np.float64,
    )
    sign_candidates = [
        np.array(signs, dtype=np.float64)
        for signs in itertools.product([-1.0, 1.0], repeat=3)
    ]
    scored = sorted(
        (
            (float(np.dot(raw_bloch * sign, target_bloch)), sign)
            for sign in sign_candidates
        ),
        key=lambda item: item[0],
        reverse=True,
    )
    return raw_bloch, scored[0][1]


noise_model = build_distillation_sim_noise_model(
    DistillationSimNoiseConfig(
        global_scalar=1.0,
        loss_scalar=0.0,
        move_duration_us=600.0,
    )
)
sim = GeminiLogicalSimulator(noise_model=noise_model)
noisy_steane7_initialize = make_noisy_steane7_initializer(sim)

kernel_bundle = build_paper_decoder_kernel_bundle(
    THETA,
    PHI,
    LAM,
    output_qubit=OUTPUT_LOGICAL,
)

actual_tasks = build_task_map(
    sim,
    kernel_bundle.actual,
    m2dets=M2DETS_5,
    m2obs=M2OBS_5,
    noisy_initializer=noisy_steane7_initialize,
    append_measurements=False,
)
special_tasks = build_task_map(
    sim,
    kernel_bundle.special,
    m2dets=M2DETS_5,
    m2obs=M2OBS_5,
    noisy_initializer=noisy_steane7_initialize,
    append_measurements=False,
)
injected_tasks = build_task_map(
    sim,
    kernel_bundle.injected,
    m2dets=M2DETS_1,
    m2obs=M2OBS_1,
    noisy_initializer=noisy_steane7_initialize,
    append_measurements=False,
)

FACTORY_TARGET = infer_factory_target(
    actual_tasks,
    shots=CONFIG['target_inference_shots'],
    output_logical=OUTPUT_LOGICAL,
    run_detectors=False,
)
DISTILLED_SIGN_VECTOR = infer_distilled_sign_vector(
    actual_tasks,
    FACTORY_TARGET,
    shots=CONFIG['target_inference_shots'],
    target_bloch=MAGIC_TARGET_BLOCH,
    output_logical=OUTPUT_LOGICAL,
    run_detectors=False,
)
INJECTED_RAW_BLOCH, INJECTED_SIGN_VECTOR = infer_single_logical_sign_vector(
    injected_tasks,
    shots=CONFIG['target_inference_shots'],
    target_bloch=MAGIC_TARGET_BLOCH,
)

print('Using output logical:', OUTPUT_LOGICAL)
print('Using factory target:', tuple(int(x) for x in FACTORY_TARGET.tolist()))
print('Distilled sign vector:', tuple(float(x) for x in DISTILLED_SIGN_VECTOR.tolist()))
print('Injected raw Bloch:', tuple(float(x) for x in INJECTED_RAW_BLOCH.tolist()))
print('Injected sign vector:', tuple(float(x) for x in INJECTED_SIGN_VECTOR.tolist()))
        


In [ ]:
print('Sampling paper-style special-circuit MLD training data...')
mld_training_data = {
    basis: run_task(
        task,
        CONFIG['mld_train_shots'],
        with_noise=True,
        run_detectors=True,
    )
    for basis, task in special_tasks.items()
}

sample_anc_det, _sample_anc_obs = split_factory_bits(
    mld_training_data['X'].detectors,
    mld_training_data['X'].observables,
    output_logical=OUTPUT_LOGICAL,
    checks_per_logical=CHECKS_PER_LOGICAL,
)
placeholder_pattern_scores = np.zeros(1 << sample_anc_det.shape[1], dtype=np.float64)

mld_decoders = {
    basis: build_mld_decoders(
        dataset,
        placeholder_pattern_scores,
        table_decoder_cls=TableDecoder,
        output_logical=OUTPUT_LOGICAL,
        checks_per_logical=CHECKS_PER_LOGICAL,
    )
    for basis, dataset in mld_training_data.items()
}

print('Sampling actual MSD score/eval data and separate injection train/eval data...')
actual_score_data = {
    basis: run_task(
        task,
        CONFIG['mld_score_shots'],
        with_noise=True,
        run_detectors=False,
    )
    for basis, task in actual_tasks.items()
}
actual_eval_data = {
    basis: run_task(
        task,
        CONFIG['eval_shots'],
        with_noise=True,
        run_detectors=False,
    )
    for basis, task in actual_tasks.items()
}
injection_training_data = {
    basis: run_task(
        task,
        CONFIG['injection_train_shots'],
        with_noise=True,
        run_detectors=False,
    )
    for basis, task in injected_tasks.items()
}
injection_eval_data = {
    basis: run_task(
        task,
        CONFIG['injection_eval_shots'],
        with_noise=True,
        run_detectors=False,
    )
    for basis, task in injected_tasks.items()
}
        


In [ ]:
print('Building MLE decoders from detector error models...')
mle_decoders = {
    basis: build_mle_decoders(
        task,
        gurobi_decoder_cls=GurobiDecoder,
        score_mode=MLE_SCORE_MODE,
        output_logical=OUTPUT_LOGICAL,
        checks_per_logical=CHECKS_PER_LOGICAL,
    )
    for basis, task in actual_tasks.items()
}

mld_curve = evaluate_mld_curve(
    actual_eval_data,
    mld_decoders,
    posterior_samples=CONFIG['posterior_samples'],
    factory_target=FACTORY_TARGET,
    sign_vector=DISTILLED_SIGN_VECTOR,
    target_bloch=MAGIC_TARGET_BLOCH,
    basis_labels=BASIS_LABELS,
    ranking_data=actual_score_data,
    output_logical=OUTPUT_LOGICAL,
    checks_per_logical=CHECKS_PER_LOGICAL,
)

mle_curve = evaluate_curve(
    actual_eval_data,
    mle_decoders,
    posterior_samples=CONFIG['posterior_samples'],
    threshold_points=CONFIG['mle_threshold_points'],
    metric='MLE logical gap',
    factory_target=FACTORY_TARGET,
    sign_vector=DISTILLED_SIGN_VECTOR,
    target_bloch=MAGIC_TARGET_BLOCH,
    basis_labels=BASIS_LABELS,
    output_logical=OUTPUT_LOGICAL,
    checks_per_logical=CHECKS_PER_LOGICAL,
)

injected_summary = injected_baseline(
    injected_tasks,
    eval_shots=CONFIG['injection_eval_shots'],
    posterior_samples=CONFIG['posterior_samples'],
    table_decoder_cls=TableDecoder,
    sign_vector=INJECTED_SIGN_VECTOR,
    target_bloch=MAGIC_TARGET_BLOCH,
    basis_labels=BASIS_LABELS,
    training_data_by_basis=injection_training_data,
    eval_data_by_basis=injection_eval_data,
    run_detectors=False,
)

print('MLE score mode:', next(iter(mle_decoders.values())).factory_score_mode)
print('Injected baseline:', injected_summary)
print('MLE curve points:', len(mle_curve['accepted_fraction']))
print('MLD curve points:', len(mld_curve['accepted_fraction']))
        


In [ ]:
plt.figure(figsize=(7, 5))

plt.plot(
    mle_curve['accepted_fraction'],
    mle_curve['fidelity'],
    marker='o',
    label='Distilled (MLE)',
)
if len(mle_curve['credible']):
    plt.fill_between(
        mle_curve['accepted_fraction'],
        mle_curve['credible'][:, 0],
        mle_curve['credible'][:, 1],
        alpha=0.15,
    )

plt.plot(
    mld_curve['accepted_fraction'],
    mld_curve['fidelity'],
    marker='o',
    label='Distilled (MLD)',
)
if len(mld_curve['credible']):
    plt.fill_between(
        mld_curve['accepted_fraction'],
        mld_curve['credible'][:, 0],
        mld_curve['credible'][:, 1],
        alpha=0.15,
    )

plt.axhline(
    injected_summary['median'],
    color='green',
    linewidth=2,
    label='Injected',
)
plt.axhspan(
    injected_summary['low'],
    injected_summary['high'],
    color='green',
    alpha=0.12,
)

plt.xlabel('Total accepted fraction')
plt.ylabel('Magic-state fidelity')
plt.title('Paper-style MSD reproduction on GeminiLogicalSimulator')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()
        


In [ ]:
{
    'factory_target': tuple(int(x) for x in FACTORY_TARGET.tolist()),
    'distilled_sign_vector': tuple(float(x) for x in DISTILLED_SIGN_VECTOR.tolist()),
    'injected_sign_vector': tuple(float(x) for x in INJECTED_SIGN_VECTOR.tolist()),
    'mle_first_points': mle_curve['fidelity'][:5].tolist(),
    'mld_first_points': mld_curve['fidelity'][:5].tolist(),
}
        
